# UP BSc Computer Science First-Year Synthetic Results Generator (2021–2025)

This notebook filters the supplied NSC synthetic datasets using the year-specific admission thresholds, constructs valid first-year curriculum pathways, generates separate semester/examination/final marks for every registered module, applies the EBIT result and progression rules, calibrates the cohort-level second-year continuation rate to the supplied cohort workbook, and saves a separate wide CSV for each year.

The 2025 CS1 replication study receives 70% of the CS1 difficulty-evidence weight and the 2006 study receives 30%. CS2023 is used to map Computer Science modules to contemporary knowledge areas.

**Path:** inputs default to `/mnt/data`. Change `DATA_DIR` in Block 1 when running elsewhere.

**Marking assumption:** exact module study guides were not supplied, so credit-bearing modules use a 50% semester-mark / 50% examination-mark formula. This is within the faculty rule that the semester/year mark contributes between 40% and 60% of the final mark.


## Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

## Paths, admissions and cohort context

In [ ]:


DATA_DIR = Path("synthetic_nsc_data")
OUTPUT_DIR = DATA_DIR / 'synthetic_first_year_cs_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 8072026
YEARS = [2021, 2022, 2023, 2024, 2025]

NSC_FILES = {year: DATA_DIR / f'NSC_Synthetic_{year}.csv' for year in YEARS}

# UP BSc Computer Science minimum requirements by entry year.
# English is Home Language OR First Additional Language.
ADMISSION_REQUIREMENTS = {
    2021: {'english_min': 60.0, 'math_min': 60.0, 'aps_min': 30},
    2022: {'english_min': 60.0, 'math_min': 60.0, 'aps_min': 30},
    2023: {'english_min': 60.0, 'math_min': 60.0, 'aps_min': 30},
    2024: {'english_min': 60.0, 'math_min': 70.0, 'aps_min': 30},
    2025: {'english_min': 60.0, 'math_min': 70.0, 'aps_min': 30},
}

PROGRAM_CODE = {
    2021: '12134001',
    2022: '12134001',
    2023: '12134001',
    2024: '12134001',
    2025: '12134002',
}

YEARBOOK_MINIMUM_CREDITS = {2021: 174, 2022: 174, 2023: 176, 2024: 176, 2025: 176}


OBSERVED_COHORT_CONTEXT = {
    2021: {'observed_year1': 304, 'observed_following_year2': 150},
    2022: {'observed_year1': 324, 'observed_following_year2': 168},
    2023: {'observed_year1': 277, 'observed_following_year2': 144},
    2024: {'observed_year1': 220, 'observed_following_year2': 130},
    2025: {'observed_year1': 286, 'observed_following_year2': 126},
}

# Scale the supplied Year 1 counts into the requested 360-420 range while
# preserving their relative ordering.
def scaled_cohort_sizes(context: Dict[int, Dict[str, int]], low: int = 360, high: int = 420) -> Dict[int, int]:
    observed = {year: values['observed_year1'] for year, values in context.items()}
    observed_min = min(observed.values())
    observed_max = max(observed.values())
    if observed_min == observed_max:
        return {year: int(round((low + high) / 2)) for year in observed}
    return {
        year: int(round(low + (count - observed_min) * (high - low) / (observed_max - observed_min)))
        for year, count in observed.items()
    }


COHORT_SIZE_BY_YEAR = scaled_cohort_sizes(OBSERVED_COHORT_CONTEXT)
TARGET_YEAR2_RATE = {
    year: values['observed_following_year2'] / values['observed_year1']
    for year, values in OBSERVED_COHORT_CONTEXT.items()
}

## Curriculum, module metadata and CS1 difficulty weights

In [ ]:

# Evidence weighting, the 2025 replication study has
# more influence than the 2006 study. The weights only adjust relative module
# difficulty; the global shift is calibrated independently to the cohort context.
RECENT_PAPER_WEIGHT = 0.70
OLDER_PAPER_WEIGHT = 0.30


def evidence_adjusted_difficulty(base: float, recent: float = 0.0, older: float = 0.0) -> float:
    return base + 5.0 * (RECENT_PAPER_WEIGHT * recent + OLDER_PAPER_WEIGHT * older)


# period: S1, S2, or YEAR
# group: controls which NSC signals influence the module mark
# recent/older: relative CS1 difficulty evidence weights, concentrated on COS modules
MODULES: Dict[str, Dict[str, object]] = {
    # Fundamental modules
    'AIM111': {'name': 'Academic information management 111', 'category': 'fundamental', 'credits': 4, 'period': 'S1', 'group': 'literacy', 'base': 0.5},
    'AIM121': {'name': 'Academic information management 121', 'category': 'fundamental', 'credits': 4, 'period': 'S2', 'group': 'literacy', 'base': 0.5},
    'ALL121': {'name': 'Academic literacy for Information Technology 121', 'category': 'fundamental', 'credits': 6, 'period': 'S2', 'group': 'literacy', 'base': 1.5},
    'UPO112': {'name': 'Academic orientation 112', 'category': 'fundamental', 'credits': 0, 'period': 'YEAR', 'group': 'completion', 'base': 0.0},

    # Common core modules
    'COS132': {'name': 'Imperative programming 132', 'category': 'core', 'credits': 16, 'period': 'S1', 'group': 'programming', 'base': 3.0, 'recent': 1.00, 'older': 0.90, 'ka': 'SDF'},
    'COS151': {'name': 'Introduction to computer science 151', 'category': 'core', 'credits': 8, 'period': 'S1', 'group': 'algorithmic', 'base': 3.5, 'recent': 0.65, 'older': 0.65, 'ka': 'AR/AL/SF'},
    'WTW115': {'name': 'Discrete structures 115', 'category': 'core', 'credits': 8, 'period': 'S1', 'group': 'math', 'base': 5.5, 'ka': 'MSF/AL'},
    'COS110': {'name': 'Program design: Introduction 110', 'category': 'core', 'credits': 16, 'period': 'S2', 'group': 'programming', 'base': 3.5, 'recent': 0.85, 'older': 1.00, 'ka': 'SDF/FPL/SE'},
    'COS122': {'name': 'Operating systems 122', 'category': 'core', 'credits': 16, 'period': 'S2', 'group': 'systems', 'base': 4.0, 'recent': 0.70, 'older': 0.50, 'ka': 'OS/SF'},

    # Core mathematics paths
    'WTW114': {'name': 'Calculus 114', 'category': 'core', 'credits': 16, 'period': 'S1', 'group': 'math', 'base': 6.5, 'ka': 'MSF'},
    'WTW124': {'name': 'Mathematics 124', 'category': 'core', 'credits': 16, 'period': 'S2', 'group': 'math', 'base': 7.5, 'ka': 'MSF'},
    'WTW162': {'name': 'Dynamical processes 162', 'category': 'core', 'credits': 8, 'period': 'S2', 'group': 'math', 'base': 6.0, 'ka': 'MSF'},
    'WTW152': {'name': 'Mathematical modelling 152', 'category': 'core', 'credits': 8, 'period': 'S1', 'group': 'math', 'base': 5.0, 'ka': 'MSF'},
    'WTW134': {'name': 'Mathematics 134', 'category': 'core', 'credits': 16, 'period': 'S1', 'group': 'math', 'base': 5.5, 'ka': 'MSF'},
    'WTW146': {'name': 'Linear algebra 146', 'category': 'core', 'credits': 8, 'period': 'S2', 'group': 'math', 'base': 5.5, 'ka': 'MSF'},
    'WTW148': {'name': 'Calculus 148', 'category': 'core', 'credits': 8, 'period': 'S2', 'group': 'math', 'base': 6.0, 'ka': 'MSF'},

    # Statistics electives
    'STK110': {'name': 'Statistics 110', 'category': 'elective', 'credits': 13, 'period': 'S1', 'group': 'statistics', 'base': 5.0, 'ka': 'MSF'},
    'STK120': {'name': 'Statistics 120', 'category': 'elective', 'credits': 13, 'period': 'S2', 'group': 'statistics', 'base': 5.5, 'ka': 'MSF'},
    'STC122': {'name': 'Statistics 122', 'category': 'elective', 'credits': 13, 'period': 'S2', 'group': 'statistics', 'base': 6.5, 'ka': 'MSF'},
    'WST111': {'name': 'Mathematical statistics 111', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'statistics', 'base': 6.5, 'ka': 'MSF'},
    'WST121': {'name': 'Mathematical statistics 121', 'category': 'elective', 'credits': 16, 'period': 'S2', 'group': 'statistics', 'base': 7.5, 'ka': 'MSF'},

    # Science electives
    'PHY114': {'name': 'First course in physics 114', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 6.5},
    'PHY124': {'name': 'First course in physics 124', 'category': 'elective', 'credits': 16, 'period': 'S2', 'group': 'science', 'base': 7.5},
    'CMY117': {'name': 'General chemistry 117', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 6.0},
    'CMY127': {'name': 'General chemistry 127', 'category': 'elective', 'credits': 16, 'period': 'S2', 'group': 'science', 'base': 6.5},
    'MLB111': {'name': 'Molecular and cell biology 111', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 4.5},
    'BOT161': {'name': 'Plants and society 161', 'category': 'elective', 'credits': 8, 'period': 'S2', 'group': 'science', 'base': 3.5},
    'MBY161': {'name': 'Introduction to microbiology 161', 'category': 'elective', 'credits': 8, 'period': 'S2', 'group': 'science', 'base': 4.0},
    'GLY155': {'name': 'Introduction to geology 155', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 4.5},
    'GLY163': {'name': 'Earth history 163', 'category': 'elective', 'credits': 16, 'period': 'S2', 'group': 'science', 'base': 5.0},
    'PHY131': {'name': 'Physics for biology students 131', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 5.0},
    'SCI154': {'name': 'Exploring the universe 154', 'category': 'elective', 'credits': 16, 'period': 'S1', 'group': 'science', 'base': 3.5},
    'ENV101': {'name': 'Introduction to environmental sciences 101', 'category': 'elective', 'credits': 8, 'period': 'S1', 'group': 'science', 'base': 3.0},
    'GGY156': {'name': 'Aspects of human geography 156', 'category': 'elective', 'credits': 8, 'period': 'S1', 'group': 'science', 'base': 3.0},
    'GGY168': {'name': 'Introduction to physical geography 168', 'category': 'elective', 'credits': 12, 'period': 'S2', 'group': 'science', 'base': 4.0},
    'GMC110': {'name': 'Cartography 110', 'category': 'elective', 'credits': 10, 'period': 'S2', 'group': 'science', 'base': 4.0},
}

for code, meta in MODULES.items():
    meta['difficulty'] = evidence_adjusted_difficulty(
        float(meta.get('base', 0.0)),
        float(meta.get('recent', 0.0)),
        float(meta.get('older', 0.0)),
    )

COMMON_MODULES = ['AIM111', 'AIM121', 'ALL121', 'UPO112', 'COS132', 'COS151', 'WTW115', 'COS110', 'COS122']
MATH_PATHS = {
    'ADVANCED_MATH': ['WTW114', 'WTW124', 'WTW162'],
    'STANDARD_MATH': ['WTW152', 'WTW134', 'WTW146', 'WTW148'],
}
STATISTICS_PATHS = {
    'STANDARD_STATISTICS': ['STK110', 'STK120'],
    'DATA_SCIENCE_STATISTICS': ['STK110', 'STC122'],
    'MATHEMATICAL_STATISTICS': ['WST111', 'WST121'],
}
SCIENCE_PATHS = {
    'PHYSICS': ['PHY114', 'PHY124'],
    'CHEMISTRY': ['CMY117', 'CMY127'],
    'BIOLOGICAL_SCIENCE': ['MLB111', 'BOT161', 'MBY161'],
    'GEOLOGY': ['GLY155', 'GLY163'],
    'ALTERNATIVE_PHYSICS': ['PHY131', 'SCI154'],
    'GEOGRAPHY': ['ENV101', 'GGY156', 'GGY168', 'GMC110'],
}

# First-year second-semester prerequisite rules.
# Admission-level Mathematics prerequisites are already enforced before sampling.
PREREQUISITES = {
    'COS110': [('pass', 'COS132'), ('pass', 'COS151')],
    'COS122': [('pass', 'COS132')],
    'WTW124': [('pass', 'WTW114')],
    'WTW162': [('pass', 'WTW114')],
    'WTW148': [('pass', 'WTW134')],
    'BOT161': [('pass', 'MLB111')],
    'CMY127': [('pass', 'CMY117')],
    'GLY163': [('pass', 'GLY155')],
    'PHY124': [('pass', 'PHY114'), ('one_pass', ('WTW114', 'WTW134'))],
    'STK120': [('pass', 'STK110')],
    'STC122': [('final_at_least', 'STK110', 60.0)],
    'WST121': [('pass', 'WST111')],
}

## Load and filter eligible NSC applicants

In [ ]:

def english_mark(frame: pd.DataFrame) -> pd.Series:
    home_is_english = frame['subject_home_language'].eq('English HL')
    fal_is_english = frame['subject_fal'].eq('English FAL')
    values = np.select(
        [home_is_english, fal_is_english],
        [frame['mark_home_language'], frame['mark_fal']],
        default=np.nan,
    )
    return pd.Series(values, index=frame.index, dtype='float64')


def mark_for_subjects(frame: pd.DataFrame, subjects: Iterable[str]) -> pd.Series:
    subjects = set(subjects)
    result = pd.Series(np.nan, index=frame.index, dtype='float64')
    for slot in (1, 2, 3):
        subject_col = f'subject_elective_{slot}'
        mark_col = f'mark_elective_{slot}'
        mask = frame[subject_col].isin(subjects)
        result.loc[mask] = frame.loc[mask, mark_col].astype(float)
    return result


def load_eligible_pool(year: int, chunksize: int = 150_000) -> pd.DataFrame:
    requirements = ADMISSION_REQUIREMENTS[year]
    eligible_chunks: List[pd.DataFrame] = []

    for chunk in pd.read_csv(NSC_FILES[year], chunksize=chunksize, low_memory=False):
        chunk['english_mark'] = english_mark(chunk)
        chunk['physical_science_mark'] = mark_for_subjects(chunk, ['Physical Sciences'])
        chunk['prior_cs_mark'] = mark_for_subjects(chunk, ['IT', 'CAT'])
        chunk['has_prior_cs_subject'] = chunk['prior_cs_mark'].notna()

        mask = (
            chunk['bachelor_eligible'].astype(bool)
            & chunk['has_mathematics'].astype(bool)
            & (chunk['mark_maths_or_ml'].astype(float) >= requirements['math_min'])
            & (chunk['english_mark'] >= requirements['english_min'])
            & (chunk['aps_score'].astype(float) >= requirements['aps_min'])
        )
        if mask.any():
            eligible_chunks.append(chunk.loc[mask].copy())

    if not eligible_chunks:
        raise ValueError(f'No eligible candidates found for {year}.')

    return pd.concat(eligible_chunks, ignore_index=True)


def sample_admitted_cohort(year: int) -> pd.DataFrame:
    pool = load_eligible_pool(year)
    n = COHORT_SIZE_BY_YEAR[year]
    if len(pool) < n:
        raise ValueError(f'Eligible pool for {year} has only {len(pool)} rows, fewer than {n}.')

    cohort = pool.sample(n=n, replace=False, random_state=MASTER_SEED + year).reset_index(drop=True)
    cohort.insert(0, 'student_id', [f'UPCS{year}{i:04d}' for i in range(1, n + 1)])
    cohort['entry_year'] = year
    cohort['program_code'] = PROGRAM_CODE[year]
    cohort['yearbook_minimum_credits'] = YEARBOOK_MINIMUM_CREDITS[year]
    cohort['admission_math_requirement'] = ADMISSION_REQUIREMENTS[year]['math_min']
    cohort['admission_english_requirement'] = ADMISSION_REQUIREMENTS[year]['english_min']
    cohort['admission_aps_requirement'] = ADMISSION_REQUIREMENTS[year]['aps_min']
    return cohort

## Assign valid curriculum pathways

In [ ]:


def sigmoid(x: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / (1.0 + np.exp(-np.asarray(x)))


def assign_curriculum_paths(cohort: pd.DataFrame, year: int) -> pd.DataFrame:
    rng = np.random.default_rng(MASTER_SEED + year * 17)
    out = cohort.copy()

    math_mark = out['mark_maths_or_ml'].astype(float).to_numpy()
    ap_math = out['has_ap_mathematics'].fillna(False).astype(bool).to_numpy()

    # Statistics pathway: mathematical statistics is concentrated among high
    # Mathematics performers; the Data Science statistics path is available in
    # every year and is made somewhat more common from 2023 onward.
    p_wst = np.clip(0.05 + 0.20 * sigmoid((math_mark - 78.0) / 5.0) + 0.08 * ap_math, 0.05, 0.32)
    p_stc_base = 0.25 if year <= 2022 else 0.32
    p_stc = np.full(len(out), p_stc_base)
    draws = rng.random(len(out))

    statistics_path = np.where(
        draws < p_wst,
        'MATHEMATICAL_STATISTICS',
        np.where(draws < p_wst + p_stc, 'DATA_SCIENCE_STATISTICS', 'STANDARD_STATISTICS'),
    )
    out['statistics_path'] = statistics_path

    # Mathematical Statistics requires the advanced Mathematics route. Other
    # students choose it with a probability increasing with NSC Mathematics.
    p_advanced_math = np.clip(0.12 + 0.58 * sigmoid((math_mark - 73.0) / 6.0) + 0.08 * ap_math, 0.12, 0.85)
    advanced_draw = rng.random(len(out))
    math_path = np.where(
        out['statistics_path'].eq('MATHEMATICAL_STATISTICS'),
        'ADVANCED_MATH',
        np.where(advanced_draw < p_advanced_math, 'ADVANCED_MATH', 'STANDARD_MATH'),
    )
    out['math_path'] = math_path

    # Science choices follow the yearbook thresholds. A Physical Sciences mark
    # below 50 is treated as not qualifying for the Level 4/5 routes and is
    # assigned to one of the routes available without qualifying Physical Science.
    physical = out['physical_science_mark'].astype(float)
    science_path: List[str] = []
    for mark in physical:
        u = rng.random()
        if pd.notna(mark) and mark >= 60.0:
            science_path.append('PHYSICS' if u < 0.55 else 'CHEMISTRY')
        elif pd.notna(mark) and mark >= 50.0:
            science_path.append('BIOLOGICAL_SCIENCE' if u < 0.60 else 'GEOLOGY')
        else:
            science_path.append('ALTERNATIVE_PHYSICS' if u < 0.35 else 'GEOGRAPHY')
    out['science_path'] = science_path

    return out


def modules_for_student(row: pd.Series) -> List[str]:
    modules = list(COMMON_MODULES)
    modules.extend(MATH_PATHS[row['math_path']])
    modules.extend(STATISTICS_PATHS[row['statistics_path']])
    modules.extend(SCIENCE_PATHS[row['science_path']])
    return list(dict.fromkeys(modules))

## Build student factors and planned module registrations

In [ ]:


def zscore(series: pd.Series, fill_value: float | None = None) -> pd.Series:
    values = series.astype(float).copy()
    if fill_value is None:
        fill_value = float(values.median())
    values = values.fillna(fill_value)
    std = float(values.std(ddof=0))
    if std == 0.0 or np.isnan(std):
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - float(values.mean())) / std


def add_hidden_student_factors(cohort: pd.DataFrame, year: int) -> pd.DataFrame:
    rng = np.random.default_rng(MASTER_SEED + year * 31)
    out = cohort.copy()

    out['_z_math'] = zscore(out['mark_maths_or_ml'])
    out['_z_english'] = zscore(out['english_mark'])
    out['_z_aps'] = zscore(out['aps_score'])
    out['_z_overall'] = zscore(out['overall_average'])
    out['_z_physical'] = zscore(out['physical_science_mark'], fill_value=float(out['physical_science_mark'].median()))
    out['_z_prior_cs'] = zscore(out['prior_cs_mark'], fill_value=0.0)

    transition_noise = rng.normal(0.0, 0.65, len(out))
    study_noise = rng.normal(0.0, 0.85, len(out))
    planning_noise = rng.normal(0.0, 0.90, len(out))

    out['_general_ability'] = (
        0.40 * out['_z_math']
        + 0.18 * out['_z_aps']
        + 0.18 * out['_z_overall']
        + 0.12 * out['_z_english']
        + 0.12 * out['_z_prior_cs']
        + transition_noise
    )
    out['_study_factor'] = 0.30 * out['_z_overall'] + study_noise
    out['_planning_factor'] = 0.25 * out['_z_math'] + 0.10 * out['_z_english'] + planning_noise
    out['_attendance_signal'] = 2.0 * sigmoid(0.60 * out['_study_factor'].to_numpy() + rng.normal(0.0, 0.55, len(out))) - 1.0

    return out


def module_ability(frame: pd.DataFrame, group: str) -> np.ndarray:
    general = frame['_general_ability'].to_numpy(float)
    z_math = frame['_z_math'].to_numpy(float)
    z_english = frame['_z_english'].to_numpy(float)
    z_physical = frame['_z_physical'].to_numpy(float)
    z_prior_cs = frame['_z_prior_cs'].to_numpy(float)
    planning = frame['_planning_factor'].to_numpy(float)

    if group == 'programming':
        return 0.48 * general + 0.27 * z_math + 0.12 * z_prior_cs + 0.13 * planning
    if group == 'algorithmic':
        return 0.52 * general + 0.33 * z_math + 0.15 * planning
    if group == 'systems':
        return 0.55 * general + 0.25 * z_math + 0.12 * z_prior_cs + 0.08 * planning
    if group == 'math':
        return 0.45 * general + 0.55 * z_math
    if group == 'statistics':
        return 0.48 * general + 0.52 * z_math
    if group == 'science':
        return 0.45 * general + 0.30 * z_physical + 0.25 * z_math
    if group == 'literacy':
        return 0.45 * general + 0.45 * z_english + 0.10 * planning
    return general


def prepare_planned_module_table(cohort: pd.DataFrame, year: int) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for _, student in cohort.iterrows():
        for code in modules_for_student(student):
            meta = MODULES[code]
            rows.append({
                'student_id': student['student_id'],
                'module_code': code,
                'module_name': meta['name'],
                'module_category': meta['category'],
                'module_credits': int(meta['credits']),
                'period': meta['period'],
                'ability_group': meta['group'],
                'difficulty': float(meta['difficulty']),
                'cs2023_knowledge_area': meta.get('ka', ''),
            })

    planned = pd.DataFrame(rows)
    factor_columns = [
        'student_id', '_general_ability', '_study_factor', '_planning_factor', '_attendance_signal',
        '_z_math', '_z_english', '_z_physical', '_z_prior_cs',
    ]
    planned = planned.merge(cohort[factor_columns], on='student_id', how='left', validate='many_to_one')

    # Fixed random shocks ensure the calibration search changes only the global
    # mark shift, not the random draw itself.
    rng = np.random.default_rng(MASTER_SEED + year * 47)
    planned['_semester_noise'] = rng.normal(0.0, 1.0, len(planned))
    planned['_exam_noise'] = rng.normal(0.0, 1.0, len(planned))
    planned['_completion_noise'] = rng.random(len(planned))

    abilities = np.empty(len(planned), dtype=float)
    for group, idx in planned.groupby('ability_group').groups.items():
        abilities[np.asarray(list(idx), dtype=int)] = module_ability(planned.loc[idx], str(group))
    planned['_module_ability'] = abilities

    return planned

## Generate marks and apply faculty rules

In [ ]:

PASS_RESULTS = {'PASS', 'DISTINCTION'}


def calculate_numeric_marks(rows: pd.DataFrame, mark_shift: float) -> Tuple[np.ndarray, np.ndarray]:
    group = rows['ability_group'].astype(str)
    study_sensitivity = group.map({
        'programming': 1.25,
        'algorithmic': 1.10,
        'systems': 1.05,
        'math': 0.90,
        'statistics': 0.95,
        'science': 0.85,
        'literacy': 0.75,
    }).fillna(0.90).to_numpy(float)

    semester = (
        60.0
        + 8.5 * rows['_module_ability'].to_numpy(float)
        + 4.5 * study_sensitivity * rows['_study_factor'].to_numpy(float)
        + 2.0 * rows['_attendance_signal'].to_numpy(float)
        - rows['difficulty'].to_numpy(float)
        + mark_shift
        + 7.5 * rows['_semester_noise'].to_numpy(float)
    )
    exam = (
        56.0
        + 10.5 * rows['_module_ability'].to_numpy(float)
        + 1.8 * rows['_study_factor'].to_numpy(float)
        + 1.2 * rows['_planning_factor'].to_numpy(float)
        - rows['difficulty'].to_numpy(float)
        + mark_shift
        + 10.5 * rows['_exam_noise'].to_numpy(float)
    )
    return np.clip(semester, 0.0, 100.0), np.clip(exam, 0.0, 100.0)


def apply_module_rules(rows: pd.DataFrame, mark_shift: float) -> pd.DataFrame:
    out = rows.copy()
    semester, exam_raw = calculate_numeric_marks(out, mark_shift)
    out['semester_mark'] = np.round(semester, 1)
    out['exam_admitted'] = out['semester_mark'] >= 40.0
    out['exam_written'] = out['exam_admitted']
    out['exam_mark'] = np.where(out['exam_admitted'], np.round(exam_raw, 1), np.nan)

    # A 50/50 split is used because the module study guides were not supplied;
    # 50% lies inside the faculty rule that the semester mark must contribute
    # between 40% and 60% of the final mark.
    out['final_mark'] = np.where(
        out['exam_admitted'],
        np.round(0.50 * out['semester_mark'] + 0.50 * out['exam_mark'], 1),
        np.nan,
    )

    conditions = [
        ~out['exam_admitted'],
        out['exam_admitted'] & (out['exam_mark'] < 40.0),
        out['exam_admitted'] & (out['exam_mark'] >= 40.0) & (out['final_mark'] < 50.0),
        out['exam_admitted'] & (out['exam_mark'] >= 40.0) & (out['final_mark'] >= 75.0),
        out['exam_admitted'] & (out['exam_mark'] >= 40.0) & (out['final_mark'] >= 50.0),
    ]
    labels = [
        'FAIL_NO_EXAM_ADMISSION',
        'FAIL_EXAM_SUBMINIMUM',
        'FAIL_FINAL_MARK',
        'DISTINCTION',
        'PASS',
    ]
    out['module_result'] = np.select(conditions, labels, default='FAIL_FINAL_MARK')
    out['module_passed'] = out['module_result'].isin(PASS_RESULTS)
    out['supplementary_candidate'] = (
        out['exam_admitted']
        & out['final_mark'].between(40.0, 49.9, inclusive='both')
    )
    return out


def prerequisite_met(student_id: str, module_code: str, s1_lookup: pd.DataFrame) -> bool:
    rules = PREREQUISITES.get(module_code, [])
    if not rules:
        return True

    for rule in rules:
        kind = rule[0]
        if kind == 'pass':
            prerequisite = str(rule[1])
            if (student_id, prerequisite) not in s1_lookup.index:
                return False
            if str(s1_lookup.loc[(student_id, prerequisite), 'module_result']) not in PASS_RESULTS:
                return False
        elif kind == 'one_pass':
            alternatives = tuple(rule[1])
            passed = False
            for prerequisite in alternatives:
                if (student_id, prerequisite) in s1_lookup.index:
                    if str(s1_lookup.loc[(student_id, prerequisite), 'module_result']) in PASS_RESULTS:
                        passed = True
                        break
            if not passed:
                return False
        elif kind == 'final_at_least':
            prerequisite = str(rule[1])
            threshold = float(rule[2])
            if (student_id, prerequisite) not in s1_lookup.index:
                return False
            mark = s1_lookup.loc[(student_id, prerequisite), 'final_mark']
            if pd.isna(mark) or float(mark) < threshold:
                return False
        else:
            raise ValueError(f'Unknown prerequisite rule: {rule}')
    return True


def simulate_results(planned: pd.DataFrame, mark_shift: float) -> pd.DataFrame:
    base_columns = [
        'student_id', 'module_code', 'module_name', 'module_category', 'module_credits',
        'period', 'ability_group', 'difficulty', 'cs2023_knowledge_area',
    ]

    s1_plan = planned.loc[planned['period'].eq('S1')].copy()
    s1_results = apply_module_rules(s1_plan, mark_shift)

    s1_credit = s1_results.loc[s1_results['module_credits'] > 0]
    any_s1_pass = s1_credit.groupby('student_id')['module_passed'].any()
    all_students = planned['student_id'].drop_duplicates()
    semester2_permitted = all_students.map(any_s1_pass).fillna(False)
    semester2_permitted.index = all_students.to_numpy()

    s1_lookup = s1_results.set_index(['student_id', 'module_code'])

    s2_plan = planned.loc[planned['period'].eq('S2')].copy()
    s2_plan['semester2_permitted'] = s2_plan['student_id'].map(semester2_permitted).fillna(False)
    s2_plan['prerequisite_met'] = [
        prerequisite_met(student_id, module_code, s1_lookup)
        for student_id, module_code in zip(s2_plan['student_id'], s2_plan['module_code'])
    ]
    eligible_s2 = s2_plan['semester2_permitted'] & s2_plan['prerequisite_met']

    s2_results_parts: List[pd.DataFrame] = []
    if eligible_s2.any():
        eligible_results = apply_module_rules(s2_plan.loc[eligible_s2].copy(), mark_shift)
        eligible_results['semester2_permitted'] = True
        eligible_results['prerequisite_met'] = True
        s2_results_parts.append(eligible_results)

    blocked = s2_plan.loc[~eligible_s2].copy()
    if not blocked.empty:
        blocked['semester_mark'] = np.nan
        blocked['exam_admitted'] = False
        blocked['exam_written'] = False
        blocked['exam_mark'] = np.nan
        blocked['final_mark'] = np.nan
        blocked['module_passed'] = False
        blocked['supplementary_candidate'] = False
        blocked['module_result'] = np.where(
            ~blocked['semester2_permitted'],
            'NOT_PERMITTED_SECOND_SEMESTER',
            'PREREQUISITE_NOT_MET',
        )
        s2_results_parts.append(blocked)

    s2_results = pd.concat(s2_results_parts, ignore_index=True) if s2_results_parts else pd.DataFrame()

    year_plan = planned.loc[planned['period'].eq('YEAR')].copy()
    if not year_plan.empty:
        completion_probability = sigmoid(4.0 + 0.55 * year_plan['_study_factor'].to_numpy(float))
        completed = year_plan['_completion_noise'].to_numpy(float) < completion_probability
        year_plan['semester_mark'] = np.nan
        year_plan['exam_admitted'] = False
        year_plan['exam_written'] = False
        year_plan['exam_mark'] = np.nan
        year_plan['final_mark'] = np.nan
        year_plan['module_result'] = np.where(completed, 'COMPLETED', 'NOT_COMPLETED')
        year_plan['module_passed'] = completed
        year_plan['supplementary_candidate'] = False
        year_plan['semester2_permitted'] = True
        year_plan['prerequisite_met'] = True

    results = pd.concat([s1_results, s2_results, year_plan], ignore_index=True, sort=False)
    results = results.sort_values(['student_id', 'period', 'module_code']).reset_index(drop=True)
    return results


def progression_summary(results: pd.DataFrame) -> pd.DataFrame:
    credit_rows = results.loc[results['module_credits'] > 0].copy()
    credit_rows['passed_credit_value'] = np.where(
        credit_rows['module_passed'], credit_rows['module_credits'], 0
    )
    credit_rows['mark_for_average'] = credit_rows['final_mark'].fillna(0.0)
    credit_rows['weighted_mark_points'] = credit_rows['mark_for_average'] * credit_rows['module_credits']

    summary = credit_rows.groupby('student_id').agg(
        planned_credits=('module_credits', 'sum'),
        passed_credits=('passed_credit_value', 'sum'),
        passed_modules=('module_passed', 'sum'),
        planned_modules=('module_code', 'count'),
        annual_weighted_mark_points=('weighted_mark_points', 'sum'),
    )
    summary['credit_completion_rate'] = summary['passed_credits'] / summary['planned_credits']
    summary['annual_credit_weighted_average'] = (
        summary['annual_weighted_mark_points'] / summary['planned_credits']
    ).round(2)
    summary['readmitted_to_second_year'] = summary['credit_completion_rate'] >= 0.70

    result_counts = (
        results.groupby(['student_id', 'module_result']).size().unstack(fill_value=0)
    )
    for column in [
        'FAIL_NO_EXAM_ADMISSION', 'FAIL_EXAM_SUBMINIMUM', 'FAIL_FINAL_MARK',
        'PREREQUISITE_NOT_MET', 'NOT_PERMITTED_SECOND_SEMESTER',
    ]:
        if column not in result_counts.columns:
            result_counts[column] = 0
    summary = summary.join(result_counts, how='left')

    s1 = results.loc[(results['period'] == 'S1') & (results['module_credits'] > 0)]
    s2_permitted = s1.groupby('student_id')['module_passed'].any().rename('semester2_permitted')
    summary = summary.join(s2_permitted, how='left')
    summary['semester2_permitted'] = summary['semester2_permitted'].fillna(False)

    summary['first_year_status'] = np.select(
        [
            ~summary['semester2_permitted'],
            summary['readmitted_to_second_year'],
        ],
        [
            'NOT_PERMITTED_SECOND_SEMESTER',
            'READMITTED_TO_SECOND_YEAR',
        ],
        default='NOT_READMITTED_BELOW_70_PERCENT_CREDITS',
    )
    return summary.reset_index()


def readmission_rate(results: pd.DataFrame) -> float:
    summary = progression_summary(results)
    return float(summary['readmitted_to_second_year'].mean())


def calibrate_mark_shift(planned: pd.DataFrame, target_rate: float) -> Tuple[float, pd.DataFrame, float]:
    low, high = -25.0, 25.0
    best_shift = 0.0
    best_results = simulate_results(planned, best_shift)
    best_rate = readmission_rate(best_results)
    best_error = abs(best_rate - target_rate)

    for _ in range(28):
        midpoint = (low + high) / 2.0
        results = simulate_results(planned, midpoint)
        rate = readmission_rate(results)
        error = abs(rate - target_rate)
        if error < best_error:
            best_shift, best_results, best_rate, best_error = midpoint, results, rate, error
        if rate < target_rate:
            low = midpoint
        else:
            high = midpoint

    return best_shift, best_results, best_rate

## Pivot to one row per student and validate outputs

In [ ]:


def make_wide_output(
    cohort: pd.DataFrame,
    results: pd.DataFrame,
    year: int,
    mark_shift: float,
    achieved_rate: float,
) -> pd.DataFrame:
    summary = progression_summary(results)
    out = cohort.merge(summary, on='student_id', how='left', validate='one_to_one')

    out['target_year2_rate'] = TARGET_YEAR2_RATE[year]
    out['synthetic_year2_rate'] = achieved_rate
    out['mark_calibration_shift'] = mark_shift
    out['observed_context_year1_count'] = OBSERVED_COHORT_CONTEXT[year]['observed_year1']
    out['observed_context_following_year2_count'] = OBSERVED_COHORT_CONTEXT[year]['observed_following_year2']
    out['synthetic_cohort_size'] = COHORT_SIZE_BY_YEAR[year]

    # Build all module columns in one DataFrame and concatenate once. This avoids
    # repeated column insertion and keeps the wide-output step efficient.
    module_columns: Dict[str, pd.Series | np.ndarray] = {}
    student_ids = out['student_id']

    for code in sorted(MODULES):
        subset = (
            results.loc[results['module_code'].eq(code)]
            .drop_duplicates('student_id')
            .set_index('student_id')
        )
        module_columns[f'{code}_registered'] = student_ids.isin(subset.index).to_numpy()

        for source, suffix in [
            ('semester_mark', 'semester_mark'),
            ('exam_mark', 'exam_mark'),
            ('final_mark', 'final_mark'),
            ('module_result', 'result'),
        ]:
            mapped = student_ids.map(subset[source])
            if suffix == 'result':
                mapped = mapped.fillna('NOT_REGISTERED')
            module_columns[f'{code}_{suffix}'] = mapped.to_numpy()

    out = pd.concat([out, pd.DataFrame(module_columns, index=out.index)], axis=1)

    # Hidden generation variables are deliberately excluded from the released dataset.
    hidden_columns = [column for column in out.columns if column.startswith('_')]
    out = out.drop(columns=hidden_columns, errors='ignore')

    # Keep identifiers and outcomes at the front, followed by source NSC fields and module columns.
    front = [
        'student_id', 'candidate_id', 'entry_year', 'program_code',
        'math_path', 'statistics_path', 'science_path',
        'planned_credits', 'passed_credits', 'credit_completion_rate',
        'semester2_permitted', 'readmitted_to_second_year', 'first_year_status',
        'annual_credit_weighted_average', 'passed_modules', 'planned_modules',
        'FAIL_NO_EXAM_ADMISSION', 'FAIL_EXAM_SUBMINIMUM', 'FAIL_FINAL_MARK',
        'PREREQUISITE_NOT_MET', 'NOT_PERMITTED_SECOND_SEMESTER',
        'target_year2_rate', 'synthetic_year2_rate', 'mark_calibration_shift',
        'synthetic_cohort_size', 'observed_context_year1_count',
        'observed_context_following_year2_count',
    ]
    front = [column for column in front if column in out.columns]
    remainder = [column for column in out.columns if column not in front]
    return out[front + remainder]


def validate_output(frame: pd.DataFrame, year: int) -> Dict[str, object]:
    requirements = ADMISSION_REQUIREMENTS[year]
    assert 360 <= len(frame) <= 420
    assert frame['bachelor_eligible'].astype(bool).all()
    assert frame['has_mathematics'].astype(bool).all()
    assert (frame['mark_maths_or_ml'] >= requirements['math_min']).all()
    assert (frame['english_mark'] >= requirements['english_min']).all()
    assert (frame['aps_score'] >= requirements['aps_min']).all()
    assert np.allclose(frame['credit_completion_rate'], frame['passed_credits'] / frame['planned_credits'])
    assert (frame['readmitted_to_second_year'] == (frame['credit_completion_rate'] >= 0.70)).all()

    # Every generated pass must satisfy exam >= 40 and final >= 50.
    for code in MODULES:
        result_col = f'{code}_result'
        exam_col = f'{code}_exam_mark'
        final_col = f'{code}_final_mark'
        if result_col not in frame:
            continue
        passed = frame[result_col].isin(PASS_RESULTS)
        if passed.any():
            assert (frame.loc[passed, exam_col] >= 40.0).all()
            assert (frame.loc[passed, final_col] >= 50.0).all()
        failed_49 = frame[result_col].eq('FAIL_FINAL_MARK') & frame[final_col].notna()
        if failed_49.any():
            assert (frame.loc[failed_49, final_col] < 50.0).all()

    return {
        'year': year,
        'rows': len(frame),
        'target_year2_rate': TARGET_YEAR2_RATE[year],
        'synthetic_year2_rate': float(frame['readmitted_to_second_year'].mean()),
        'semester2_permitted_rate': float(frame['semester2_permitted'].mean()),
        'mean_passed_credit_rate': float(frame['credit_completion_rate'].mean()),
        'mean_annual_weighted_average': float(frame['annual_credit_weighted_average'].mean()),
        'mean_planned_credits': float(frame['planned_credits'].mean()),
    }

## Generate and save the five yearly CSV files

In [ ]:


def generate_year(year: int, save_long: bool = False) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, object]]:
    cohort = sample_admitted_cohort(year)
    cohort = assign_curriculum_paths(cohort, year)
    cohort = add_hidden_student_factors(cohort, year)
    planned = prepare_planned_module_table(cohort, year)

    shift, results, achieved_rate = calibrate_mark_shift(planned, TARGET_YEAR2_RATE[year])
    wide = make_wide_output(cohort, results, year, shift, achieved_rate)
    validation = validate_output(wide, year)

    wide_path = OUTPUT_DIR / f'UP_BScCS_FirstYear_Results_{year}.csv'
    wide.to_csv(wide_path, index=False)

    if save_long:
        long_path = OUTPUT_DIR / f'UP_BScCS_FirstYear_ModuleResults_Long_{year}.csv'
        results.to_csv(long_path, index=False)

    return wide, results, validation


def save_metadata() -> None:
    metadata_rows = []
    for code, meta in sorted(MODULES.items()):
        metadata_rows.append({
            'module_code': code,
            'module_name': meta['name'],
            'module_category': meta['category'],
            'credits': meta['credits'],
            'period': meta['period'],
            'ability_group': meta['group'],
            'difficulty_points': round(float(meta['difficulty']), 3),
            'recent_paper_weight_component': float(meta.get('recent', 0.0)),
            'older_paper_weight_component': float(meta.get('older', 0.0)),
            'cs2023_knowledge_area': meta.get('ka', ''),
        })
    pd.DataFrame(metadata_rows).to_csv(OUTPUT_DIR / 'UP_BScCS_Module_Metadata.csv', index=False)


def save_module_outcome_summary() -> None:
    rows = []
    non_attempt_results = {
        'PREREQUISITE_NOT_MET',
        'NOT_PERMITTED_SECOND_SEMESTER',
        'NOT_REGISTERED',
        'COMPLETED',
        'NOT_COMPLETED',
    }

    for year in YEARS:
        frame = pd.read_csv(
            OUTPUT_DIR / f'UP_BScCS_FirstYear_Results_{year}.csv',
            low_memory=False,
        )
        for code, meta in sorted(MODULES.items()):
            registered_col = f'{code}_registered'
            result_col = f'{code}_result'
            semester_col = f'{code}_semester_mark'
            exam_col = f'{code}_exam_mark'
            final_col = f'{code}_final_mark'

            registered = frame.loc[frame[registered_col].fillna(False).astype(bool)]
            attempted = registered.loc[~registered[result_col].isin(non_attempt_results)]
            passes = int(attempted[result_col].isin(PASS_RESULTS).sum())
            failures = int(len(attempted) - passes)

            rows.append({
                'year': year,
                'module_code': code,
                'module_name': meta['name'],
                'module_category': meta['category'],
                'registered_students': len(registered),
                'attempted_students': len(attempted),
                'passes': passes,
                'failures': failures,
                'pass_rate': passes / len(attempted) if len(attempted) else np.nan,
                'fail_rate': failures / len(attempted) if len(attempted) else np.nan,
                'mean_semester_mark': attempted[semester_col].mean() if len(attempted) else np.nan,
                'mean_exam_mark': attempted[exam_col].mean() if len(attempted) else np.nan,
                'mean_final_mark': attempted[final_col].mean() if len(attempted) else np.nan,
                'difficulty_points': round(float(meta['difficulty']), 3),
                'cs2023_knowledge_area': meta.get('ka', ''),
            })

    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / 'UP_BScCS_Module_Outcome_Summary.csv',
        index=False,
    )


def main() -> None:
    summary_rows = []
    for year in YEARS:
        print(f'Generating {year} with {COHORT_SIZE_BY_YEAR[year]} students...')
        _, _, validation = generate_year(year, save_long=False)
        summary_rows.append(validation)
        print(
            f"  target Year-2 rate={validation['target_year2_rate']:.3f}; "
            f"synthetic={validation['synthetic_year2_rate']:.3f}"
        )

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(OUTPUT_DIR / 'UP_BScCS_FirstYear_Generation_Summary.csv', index=False)
    save_metadata()
    save_module_outcome_summary()
    print('\nSaved files to:', OUTPUT_DIR)
    print(summary.to_string(index=False))


if __name__ == '__main__':
    main()